# This Notebook estimates the model

## Settings

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
from scipy.optimize import minimize

import DynamicTimeAllocationModel

# c++ settings
do_compile = True
threads = 64

from EconModel import cpptools
cpptools.setup_nlopt(folder='cppfuncs/', do_print=True)

NLopt already installed


In [2]:
# setup model
settings = { 
       # technical settings
       'threads':threads,
       'do_multistart': False,
       'do_egm': True,
       'interp_method': 'linear',
       'interp_inverse': True,
       'precompute_intratemporal': True,
       'centered_gradient': True,
       'bargaining': 'limited',
}


model = DynamicTimeAllocationModel.HouseholdModelClass(par=settings) 
model.link_to_cpp(force_compile=do_compile)

## Empirical Moments to Match

In [3]:
# all moments listed here will be used in estimation. Comment out those you do not want to use.
datamoms = dict()

# wages
datamoms['wage_level_w_25_34'] = 40.1
datamoms['wage_level_m_25_34'] = 49.3
datamoms['wage_level_w_35_41'] = 50.4
datamoms['wage_level_m_35_41'] = 67.8

# employment rates
datamoms['employment_rate_w_35_41'] = 64.0 # 70.0
datamoms['employment_rate_m_35_41'] = 88.0 # 85.0
datamoms['work_hours_w'] = 1674.0 / 52.0  # annual hours to weekly hours
datamoms['work_hours_m'] = 2062.0 / 52.0  # annual hours to weekly hours

# # consumption
datamoms['consumption'] = 42.716
datamoms['consumption_90_10_ratio'] = 3.33 * 1.0954

# # marriage and divorce rates
datamoms['marriage_rate_35_41'] = 69.0

# Mazzocco moments
datamoms['home_prod_w'] = 1535.0 / 52
datamoms['home_prod_m'] = 1035.0 / 52


# weights
weights = dict()
for mom in ('consumption_90_10_ratio',):
    weights[mom] = 10.0
    

## Parameters to estimate

In [4]:
# parameters to estimate
estpars = {
    # Wages
    'mu': {'guess':2.3678,'lower':0.1,'upper':3.00}, 
    'mu_mult': {'guess':1.1126,'lower':1.0,'upper':3.0},
    'gamma': {'guess':0.1237,'lower':0.001,'upper':0.50},
    'gamma_mult': {'guess':1.7611,'lower':1.0,'upper':3.0},
    'sigma_mu': {'guess':0.5613,'lower':0.001,'upper':1.0},
    
    # Disutility from work
    'eta': {'guess':0.9033,'lower':0.1,'upper':5.0},
    'eta_mult': {'guess':0.8877,'lower':0.3,'upper':3.0},
    'phi': {'guess':4.4732,'lower':0.1,'upper':5.0},
    'phi_mult': {'guess':1.0855,'lower':0.3,'upper':3.0},
    
    # Home production
    'alpha': {'guess':0.9608,'lower':0.1,'upper':1.9},
    'pi': {'guess':0.6144,'lower':0.1,'upper':0.9},
    'lambda_': {'guess':5.7527,'lower':0.1,'upper':30.0},
    
    # # Match quality
    'sigma_love': {'guess':3.7895,'lower':0.01,'upper':20.5},
}

## setup initial guess 

In [5]:
# check bounds
bounds_ok = True
for key in estpars.keys():
    if estpars[key]['guess']<estpars[key]['lower']:
        print(key,' lower',estpars[key]['guess'])
        bounds_ok = False
    
    if estpars[key]['guess']>estpars[key]['upper']:
        print(key,' upper',estpars[key]['guess'])
        bounds_ok = False

if not bounds_ok:
    stop

In [6]:
# check initial guess
theta_init = np.array([estpars[key]['guess'] for key in estpars.keys()])
obj_init = model.obj_func(theta_init, estpars, datamoms, weights, do_print=True)

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6246, data: 40.1000
  wage_level_m_25_34       : sim: 50.0506, data: 49.3000
  wage_level_w_35_41       : sim: 51.8620, data: 50.4000
  wage_level_m_35_41       : sim: 67.0226, data: 67.8000
  employment_rate_w_35_41  : sim: 63.8269, data: 64.0000
  employment_rate_m_35_41  : sim: 88.4375, data: 88.0000
  work_hours_w             : sim: 29.0932, data: 32.1923
  work_hours_m             : sim: 36.3681, data

## Estimate model

In [7]:
# Estimate model using nelder-mead algorithm
do_print = True
res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',
               options={'xatol': 1e-3, 'fatol': 1e-3, 'disp': True, 'maxiter':500, 'maxfev':500})


Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.6246, data: 40.1000
  wage_level_m_25_34       : sim: 50.0506, data: 49.3000
  wage_level_w_35_41       : sim: 51.8620, data: 50.4000
  wage_level_m_35_41       : sim: 67.0226, data: 67.8000
  employment_rate_w_35_41  : sim: 63.8269, data: 64.0000
  employment_rate_m_35_41  : sim: 88.4375, data: 88.0000
  work_hours_w             : sim: 29.0932, data: 32.1923
  work_hours_m             : sim: 36.3681, data

Parameters:
  mu             : 2.4862 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.6811, data: 40.1000
  wage_level_m_25_34       : sim: 53.4892, data: 49.3000
  wage_level_w_35_41       : sim: 58.6674, data: 50.4000
  wage_level_m_35_41       : sim: 74.3338, data: 67.8000
  employment_rate_w_35_41  : sim: 62.1446, data: 64.0000
  employment_rate_m_35_41  : sim: 93.4340, data: 88.0000
  work_hours_w             : sim: 28.6434, data: 32.1923
  work_hours_m             : sim: 38.1190, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1682 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 43.3469, data: 40.1000
  wage_level_m_25_34       : sim: 53.2299, data: 49.3000
  wage_level_w_35_41       : sim: 53.6991, data: 50.4000
  wage_level_m_35_41       : sim: 73.4069, data: 67.8000
  employment_rate_w_35_41  : sim: 52.2344, data: 64.0000
  employment_rate_m_35_41  : sim: 95.0166, data: 88.0000
  work_hours_w             : sim: 26.4359, data: 32.1923
  work_hours_m             : sim: 38.4905, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1299 (init: 0.1237)
  gamma_mult     : 1.7611 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.9453, data: 40.1000
  wage_level_m_25_34       : sim: 49.2425, data: 49.3000
  wage_level_w_35_41       : sim: 52.7332, data: 50.4000
  wage_level_m_35_41       : sim: 67.6253, data: 67.8000
  employment_rate_w_35_41  : sim: 62.8817, data: 64.0000
  employment_rate_m_35_41  : sim: 91.8477, data: 88.0000
  work_hours_w             : sim: 29.0332, data: 32.1923
  work_hours_m             : sim: 37.3221, data

Parameters:
  mu             : 2.3678 (init: 2.3678)
  mu_mult        : 1.1126 (init: 1.1126)
  gamma          : 0.1237 (init: 0.1237)
  gamma_mult     : 1.8492 (init: 1.7611)
  sigma_mu       : 0.5613 (init: 0.5613)
  eta            : 0.9033 (init: 0.9033)
  eta_mult       : 0.8877 (init: 0.8877)
  phi            : 4.4732 (init: 4.4732)
  phi_mult       : 1.0855 (init: 1.0855)
  alpha          : 0.9608 (init: 0.9608)
  pi             : 0.6144 (init: 0.6144)
  lambda_        : 5.7527 (init: 5.7527)
  sigma_love     : 3.7895 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 38.3878, data: 40.1000
  wage_level_m_25_34       : sim: 48.7553, data: 49.3000
  wage_level_w_35_41       : sim: 52.3175, data: 50.4000
  wage_level_m_35_41       : sim: 67.3053, data: 67.8000
  employment_rate_w_35_41  : sim: 61.5599, data: 64.0000
  employment_rate_m_35_41  : sim: 92.7058, data: 88.0000
  work_hours_w             : sim: 28.6545, data: 32.1923
  work_hours_m             : sim: 37.5987, data

...

Parameters:
  mu             : 2.3810 (init: 2.3678)
  mu_mult        : 1.1048 (init: 1.1126)
  gamma          : 0.1186 (init: 0.1237)
  gamma_mult     : 1.8519 (init: 1.7611)
  sigma_mu       : 0.5591 (init: 0.5613)
  eta            : 0.9084 (init: 0.9033)
  eta_mult       : 0.8930 (init: 0.8877)
  phi            : 4.5215 (init: 4.4732)
  phi_mult       : 1.1133 (init: 1.0855)
  alpha          : 0.9642 (init: 0.9608)
  pi             : 0.6122 (init: 0.6144)
  lambda_        : 5.7887 (init: 5.7527)
  sigma_love     : 3.8289 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.7671, data: 40.1000
  wage_level_m_25_34       : sim: 49.9612, data: 49.3000
  wage_level_w_35_41       : sim: 51.7896, data: 50.4000
  wage_level_m_35_41       : sim: 67.0132, data: 67.8000
  employment_rate_w_35_41  : sim: 64.1794, data: 64.0000
  employment_rate_m_35_41  : sim: 88.3812, data: 88.0000
  work_hours_w             : sim: 29.1292, data: 32.1923
  work_hours_m             : sim: 36.3470, data

Parameters:
  mu             : 2.3811 (init: 2.3678)
  mu_mult        : 1.1049 (init: 1.1126)
  gamma          : 0.1185 (init: 0.1237)
  gamma_mult     : 1.8525 (init: 1.7611)
  sigma_mu       : 0.5591 (init: 0.5613)
  eta            : 0.9085 (init: 0.9033)
  eta_mult       : 0.8931 (init: 0.8877)
  phi            : 4.5216 (init: 4.4732)
  phi_mult       : 1.1136 (init: 1.0855)
  alpha          : 0.9640 (init: 0.9608)
  pi             : 0.6121 (init: 0.6144)
  lambda_        : 5.7886 (init: 5.7527)
  sigma_love     : 3.8291 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.7658, data: 40.1000
  wage_level_m_25_34       : sim: 49.9723, data: 49.3000
  wage_level_w_35_41       : sim: 51.7839, data: 50.4000
  wage_level_m_35_41       : sim: 67.0236, data: 67.8000
  employment_rate_w_35_41  : sim: 64.1827, data: 64.0000
  employment_rate_m_35_41  : sim: 88.3731, data: 88.0000
  work_hours_w             : sim: 29.1295, data: 32.1923
  work_hours_m             : sim: 36.3442, data

Parameters:
  mu             : 2.3810 (init: 2.3678)
  mu_mult        : 1.1048 (init: 1.1126)
  gamma          : 0.1186 (init: 0.1237)
  gamma_mult     : 1.8524 (init: 1.7611)
  sigma_mu       : 0.5591 (init: 0.5613)
  eta            : 0.9085 (init: 0.9033)
  eta_mult       : 0.8931 (init: 0.8877)
  phi            : 4.5220 (init: 4.4732)
  phi_mult       : 1.1136 (init: 1.0855)
  alpha          : 0.9641 (init: 0.9608)
  pi             : 0.6121 (init: 0.6144)
  lambda_        : 5.7891 (init: 5.7527)
  sigma_love     : 3.8287 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.7643, data: 40.1000
  wage_level_m_25_34       : sim: 49.9666, data: 49.3000
  wage_level_w_35_41       : sim: 51.7842, data: 50.4000
  wage_level_m_35_41       : sim: 67.0203, data: 67.8000
  employment_rate_w_35_41  : sim: 64.1845, data: 64.0000
  employment_rate_m_35_41  : sim: 88.3760, data: 88.0000
  work_hours_w             : sim: 29.1300, data: 32.1923
  work_hours_m             : sim: 36.3453, data

Parameters:
  mu             : 2.3811 (init: 2.3678)
  mu_mult        : 1.1048 (init: 1.1126)
  gamma          : 0.1185 (init: 0.1237)
  gamma_mult     : 1.8524 (init: 1.7611)
  sigma_mu       : 0.5591 (init: 0.5613)
  eta            : 0.9085 (init: 0.9033)
  eta_mult       : 0.8931 (init: 0.8877)
  phi            : 4.5214 (init: 4.4732)
  phi_mult       : 1.1136 (init: 1.0855)
  alpha          : 0.9640 (init: 0.9608)
  pi             : 0.6122 (init: 0.6144)
  lambda_        : 5.7887 (init: 5.7527)
  sigma_love     : 3.8298 (init: 3.7895)
Moments:
  wage_level_w_25_34       : sim: 37.7650, data: 40.1000
  wage_level_m_25_34       : sim: 49.9686, data: 49.3000
  wage_level_w_35_41       : sim: 51.7843, data: 50.4000
  wage_level_m_35_41       : sim: 67.0158, data: 67.8000
  employment_rate_w_35_41  : sim: 64.1826, data: 64.0000
  employment_rate_m_35_41  : sim: 88.3730, data: 88.0000
  work_hours_w             : sim: 29.1299, data: 32.1923
  work_hours_m             : sim: 36.3448, data

C:\Users\zbk883\AppData\Local\Temp\4\ipykernel_5592\1613049469.py:3: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  res = minimize(model.obj_func, theta_init, args=(estpars, datamoms,weights,do_print), method='Nelder-Mead',


In [8]:
model.save_par('calibrated_par')